# Tutorial on the usage of the RSA tool for Proteins

This tutorial illustrates how to use the Ring Stacking Analysis (RSA) tool on proteins. This tutorial is divided into three parts. 
1. Editing a protein structure file so that the RSA tool can correctly identify macromolecules.
2. Running the optimized RSA tool.
3. Extracting the connected networks and exporting them as PDB files.

Before starting any analysis, load the necessary modules and define the file paths.

In [ ]:
import os
import numpy as np
import pandas as pd
import MDAnalysis as mda

from pysoftk.pol_analysis.ring_ring import RSA

# Ensure data directories exist
os.makedirs('data', exist_ok=True)
network_pdb_dir = 'data/network_pdbs'
os.makedirs(network_pdb_dir, exist_ok=True)

## 1. Editing the protein structure file
The PySoftK RSA tool identifies distinct macromolecules by their MDAnalysis `resid`. In a standard protein PDB, the `resid` denotes individual amino acids and resets for each chain. To use the RSA tool correctly, **all atoms of the same protein must share a single, unique `resid`**.

In [ ]:
original_topology = 'data/protein_rsa_noedit.pdb'
trajectory = 'data/trajectory_rsa_protein.xtc'
edited_topology = 'data/trajectory_resids.pdb'

print(f"Loading Universe: {original_topology}")
u = mda.Universe(original_topology, trajectory)

The following code dynamically iterates through the distinct chains (`segid`) in your system and assigns a single, unique `resid` to each entire protein.

In [ ]:
proteins = u.select_atoms('protein')
protein_pos = proteins.positions
proteins.positions = protein_pos 

# Assign a unique resid to each protein chain (segid)
segids = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']
new_resids = []

for i, seg in enumerate(segids, start=1):
    protein_seg = u.select_atoms(f'segid {seg}')
    seg_len = len(np.unique(protein_seg.resids))
    new_resids.extend([i] * seg_len)

print(f"Total new resids generated: {len(new_resids)}")
print(f"Total protein atoms: {len(proteins)}")

# Overwrite the topology's resids with our new unique identifiers
proteins.residues.resids = new_resids

# Save the properly formatted structure
with mda.Writer(edited_topology, proteins.n_atoms) as W:
    W.write(proteins)

print(f"Edited topology saved to: {edited_topology}")

## 2. Running RSA 
With the new structure file, we can run the highly-optimized RSA tool in the exact same way we run it on polymers.

In [ ]:
# Name output file
results_file = 'data/rsa_prot_tutorial.parquet'

# Calculation parameters
ang_c = 30
dist_c = 5
start, stop, step = 0, 20, 2

In [ ]:
# Initialize RSA with the NEW edited topology
rsa_calc = RSA(edited_topology, trajectory)

print("Ring Stacking analysis has started...")
# write_pdb=False prevents cluttering the folder with individual pairs
rsa_calc.stacking_analysis(dist_c, ang_c, start, stop, step, results_file, write_pdb=False)
print("Stacking analysis complete.")

## 3. Exploring the Results
The output is stored in a Pandas DataFrame. Because we are analyzing entire macromolecules, we can safely drop the redundant `atom_index` column. The remaining `pol_resid` column contains pairs of our newly assigned Protein IDs.

In [ ]:
df = pd.read_parquet(results_file)

# CLEANUP: Drop the 'atom_index' column since the new RSA tool 
# optimizes by using 'pol_resid' exclusively.
if 'atom_index' in df.columns:
    df = df.drop(columns=['atom_index'])

df.head()

In [ ]:
print("Example of interacting protein pairs in the first recorded stack:")
print(df['pol_resid'].iloc[0])

## 4. Network Analysis & PDB Generation
The RSA tool uses graph theory to evaluate the pairs and identify continuous clusters of proteins interconnected through $\pi$-$\pi$ stacking. The following code extracts these networks and saves them as visualizable `.pdb` files.

In [ ]:
# Extract the network groupings
sev_ring = rsa_calc.find_several_rings_stacked(results_file)

# Grab the universe from our RSA calculator and set it to the first frame
u_rsa = rsa_calc.get_mda_universe()
u_rsa.trajectory[0] 

print("Connected Networks (Residue IDs of stacked proteins):")
if sev_ring and len(sev_ring) > 0:
    for i, network in enumerate(sev_ring[0], start=1):
        print(f"\n  Cluster {i} members: {network}")
        
        # Convert the set of resids into an MDAnalysis selection string
        resid_str = " ".join(map(str, network))
        cluster_selection = u_rsa.select_atoms(f'resid {resid_str}')
        
        # Define the output path and write the PDB
        pdb_filename = os.path.join(network_pdb_dir, f"network_cluster_{i}.pdb")
        with mda.Writer(pdb_filename, cluster_selection.n_atoms) as W:
            W.write(cluster_selection)
            
        print(f"  -> Saved structure to: {pdb_filename}")
else:
    print("  No networks found.")
    
print("\nWorkflow complete!")